In [1]:
import numpy as np
from scipy.spatial import Voronoi,voronoi_plot_2d
import random as rnd
from IPython.display import display, HTML,Javascript,clear_output
from svg_plot import SVGPlot


In [2]:
colors = [
    "rgb(255,50,50)",    # Bright Red
    "rgb(50,255,50)",    # Bright Green
    "rgb(50,50,255)",    # Bright Blue
    "rgb(255,255,50)",   # Bright Yellow
    "rgb(255,50,255)",   # Bright Magenta
    "rgb(50,255,255)",   # Bright Cyan
    "rgb(200,50,50)",    # Dark Red
    "rgb(50,200,50)",    # Dark Green
    "rgb(50,50,200)",    # Dark Blue
    "rgb(200,100,50)",   # Orange
    "rgb(255,100,150)",  # Pink/Red
    "rgb(150,50,255)",   # Purple
    "rgb(50,150,255)",   # Sky Blue
    "rgb(255,150,50)",   # Orange-Yellow
    "rgb(100,255,100)",  # Light Green (but saturated)
    "rgb(255,100,255)",  # Bright Pink
    "rgb(180,50,180)",   # Deep Purple
    "rgb(50,180,180)",   # Teal
    "rgb(180,180,50)",   # Olive
    "rgb(150,50,100)"    # Maroon/Burgundy
]
NUMBER_OF_AREAS = 60

In [3]:
def point_in_polygon(x, y, polygon):
    polygon = np.asarray(polygon)
    n = len(polygon)
    inside = False
    
    for i in range(n):
        x1, y1 = polygon[i]
        x2, y2 = polygon[(i + 1) % n]
        
        # Check if edge crosses horizontal ray to the right
        if ((y1 > y) != (y2 > y)) and (x < (x2 - x1) * (y - y1) / (y2 - y1) + x1):
            inside = not inside
    
    return inside

In [4]:
class PopulationArea:
    def __init__(self,area_id,center,vertices):
        self.area_id = area_id
        self.center = center
        self.vertices = vertices
        self.local_population = []

In [ ]:
def get_area(areas,x,y):
    

In [ ]:
def linear_distance_prob(x):
    x = np.array(x)
    prob = np.zeros_like(x, dtype=float)
    mask = (x >= 0) & (x <= 10)
    
    if np.any(mask):
        prob[mask] = 0.2 - 0.02 * x[mask]
        
    return prob

In [5]:
def probability_curve(x, start_point=20, end_point=70, decay_rate=None):
    if x <20 :
        return 0.0
    x = np.array(x) if hasattr(x, '__iter__') else np.array([x])
    if decay_rate is None:
        decay_rate = -np.log(0.01) / (end_point - start_point)
    shifted_x = x - start_point
    prob = np.exp(-decay_rate * shifted_x)
    prob = np.clip(prob, 0, 1)
    if len(prob) == 1:
        return prob[0]
    return prob

In [6]:
def generate_age():
    def age_probability(age):
        if age < 0:
            return 0
        
        # Peak at ~38, rise then fall
        peak = 38
        rise = (age / peak) ** 1.5
        fall = np.exp(-0.015 * (age - peak))
        
        return rise * fall
    age = -1
    while age < 0:
        candidate = np.random.uniform(0, 100)
        p = age_probability(candidate)
        max_p = age_probability(38)  # Peak probability
        
        if np.random.rand() < (p / max_p):
            return int(candidate)


In [11]:
class Denizen:
    def __init__(self, area):
        self.area = area
        self.pos = (0, 0)
        self.age = generate_age()
        self.moved = False

    def assign_pos(self,):
        # FIXED: Extract correct coordinates
        x_vals = [point[0] for point in self.area.vertices]  # x is first element
        y_vals = [point[1] for point in self.area.vertices]  # y is second element
        
        x_min, x_max = min(x_vals), max(x_vals)
        y_min, y_max = min(y_vals), max(y_vals)
        
        while True:
            # FIXED: Correct random range
            x = x_min + rnd.random() * (x_max - x_min)
            y = y_min + rnd.random() * (y_max - y_min)
            
            if point_in_polygon(x, y, self.area.vertices):
                self.pos = (float(x), float(y))
                self.area.local_population.append(self)
                return

    def __str__(self):
        return f"Vector({self.area.area_id}, {self.pos},{self.age})"

In [12]:
seed = 95#rnd.randint(0,100)
print(seed)
np.random.seed(seed)  # For reproducible results
points = np.random.rand(NUMBER_OF_AREAS, 2) * 10  # NUMBER_OF_AREAS rows, 2 columns, scale to 0-10
vor = Voronoi(points)

95


In [13]:
areas = []
area_id = 0;
for i in range(len(points)):
    region_idx = vor.point_region[i]
    region = vor.regions[region_idx]
    if region and -1 not in region: #if region` - Ensures the region is not empty -1 not in region` - Filters out unbounded regions
        vertices = vor.vertices[region]
        x_min, y_min = vertices.min(axis=0)  # [min_x, min_y]
        x_max, y_max = vertices.max(axis=0)  # [max_x, max_y]
        if(max([x_max,y_max])<10) and (min([x_min,y_min])>0):
            areas.append(PopulationArea(area_id,points[i],vertices))
            area_id +=1

In [22]:
population=[]
# for area in areas:
for _ in range(400):
    person = Denizen(areas[5])
    person.assign_pos()
    population.append(person)

In [23]:
print(population[10])

Vector(5, (2.535930721285228, 3.4327017166357248),77)


In [29]:
svg = SVGPlot(500, 500, 10)

output_display = display(HTML("<div id='sim-container'>Loading...</div>"), display_id=True)
svg.clear()
for area in areas:
     svg.add_polygon(area.vertices,'rgb(220,220,220)')
for i in range(len(population)):
    p = population[i]
    x = p.pos[0]
    y = p.pos[1]
    svg.add_rect(x,y,f'rgb(522,{p.age*3},{p.age*3})',0.05)
    
new_svg = svg.get_canvas()
output_display.update(HTML(f"<div id='sim-container'>{new_svg}</div>"))
